# UFC Fight Prediction and Betting Market Evaluation

## Core Research Question

Can historical UFC fight data be used to build a probabilistic model for fight outcomes, and can those probabilities identify betting opportunities that the market has mispriced?

## Project Process

This project moved through six main stages:

1. **Data cleaning and leakage prevention**
   - Standardized fighter and fight-level data
   - Built pre-fight cumulative, recency, Elo, style, cardio, and matchup features
   - Explicitly excluded current-fight and post-fight leakage from modeling inputs

2. **Validation-first model development**
   - Used chronological train/validation/test structure
   - Compared interpretable logistic regression specifications first
   - Tested grouped hypotheses, ablations, and recency/feature-selection ideas on validation only

3. **Frozen final benchmark models**
   - Finalized a lean common feature set
   - Evaluated frozen Logistic Regression, Random Forest, and XGBoost once on the hold-out test set
   - Found that Logistic Regression had the best test log loss / ROC AUC / Brier score, while Random Forest had the best accuracy

4. **Odds ingestion and merge**
   - Cleaned the Kaggle UFC odds dataset
   - Selected a single source rather than averaging books
   - Converted odds to no-vig implied probabilities and merged them into the modeling dataset

5. **Exploratory betting backtest**
   - Uses frozen Logistic Regression test probabilities as the main post-cutoff betting input
   - Retains the older Random Forest betting pass as a historical/exploratory comparison
   - Compares model probabilities to market no-vig probabilities
   - Simulates flat-bet and fractional Kelly staking
   - Finds that both betting passes remain negative across tested thresholds

6. **Walk-forward market research**
   - Built a separate pre-test walk-forward workflow on the odds-matched train+validation era
   - Used annual expanding folds with a one-year calibration slice and next-year out-of-sample test slice
   - Compared raw Random Forest probabilities, calibrated variants, market no-vig probabilities, and model-market blends
   - Found that calibration helped the model somewhat, blends came close to the market, but the market remained the strongest standalone probability benchmark

## What the project shows

- Careful leakage-aware feature engineering
- Time-aware model validation
- Model-family comparison under a common feature set
- Probability evaluation, not just classification accuracy
- Market-relative benchmarking and post-hoc betting diagnostics
- Auditability and reproducibility checks

## Main conclusion so far

The project shows that UFC fight outcomes are meaningfully predictable, but profitable betting is much harder than pure classification. The market already captures a large share of the available information. The most promising research direction is therefore not just better winner prediction, but better calibration, cleaner market benchmarking, and stronger tests of whether the model adds signal beyond market implied probabilities.


## Important Methodology Update

During final audit and EDA review, the project identified a structural break in the historical source data: pre-2010 fights show implausibly high red-corner win rates, which strongly suggests the older rows do not preserve a stable red/blue corner convention.

From that point forward, the repo treats `2010-01-01` as the start of the reliable modeling era. The full historical table is still kept for descriptive analysis, but all benchmark modeling, odds matching, betting analysis, and walk-forward research now use the post-2009 subset only.


In [ ]:
from pathlib import Path
import pandas as pd

repo_root = Path.cwd()
if repo_root.name == "notebooks":
    repo_root = repo_root.parent

final_comparison = pd.read_csv(repo_root / "outputs" / "final_model_test_comparison.csv")
walk_forward_comparison = pd.read_csv(repo_root / "outputs" / "walk_forward_model_comparison.csv")
betting_comparison = pd.read_csv(repo_root / "outputs" / "betting_model_backtest_comparison.csv")

print("Final frozen hold-out model comparison:")
display(final_comparison)

print("Main-vs-historical betting model comparison:")
display(betting_comparison)

print("Pre-test walk-forward market benchmark comparison:")
display(walk_forward_comparison)
